In [1]:
import pandas as pd


df = pd.read_json("archive/IMDB_reviews.json", lines=True)


df.head()

,review_date,movie_id,user_id,is_spoiler,review_text,rating,review_summary
0,10 February 2006,tt0111161,ur1898687,True,"In its Oscar year, Shawshank Redemption (writt...",10,A classic piece of unforgettable film-making.
1,6 September 2000,tt0111161,ur0842118,True,The Shawshank Redemption is without a doubt on...,10,Simply amazing. The best film of the 90's.
2,3 August 2001,tt0111161,ur1285640,True,I believe that this film is the best story eve...,8,The best story ever told on film
3,1 September 2002,tt0111161,ur1003471,True,"**Yes, there are SPOILERS here**This film has ...",10,Busy dying or busy living?
4,20 May 2004,tt0111161,ur0226855,True,At the heart of this extraordinary movie is a ...,8,"Great story, wondrously told and acted"


In [2]:

print("データのサイズ:", df.shape)


df = df[['review_text', 'is_spoiler']]


print("\nネタバレの割合:")
print(df['is_spoiler'].value_counts())

データのサイズ: (573913, 7)

ネタバレの割合:
is_spoiler
False    422989
True     150924
Name: count, dtype: int64


In [11]:

df = df.dropna()


df_sample = df.sample(n=10000, random_state=42).copy()

print("実験用データのサイズ:", df_sample.shape)

実験用データのサイズ: (500000, 2)


In [12]:
from sklearn.metrics import accuracy_score

# 1. 自分でNGワード辞書（ルール）を作る
# ※ここは英語のレビューなので英語の単語を指定します
ng_words = ["die", "kill", "death", "ending", "secret", "plot twist", "spoiler"]

# 2. ルールベースで判定する関数を作成
def check_spoiler_rule(text):
    text_lower = text.lower() # 大文字・小文字のブレを無くすため、全て小文字に変換
    for word in ng_words:
        if word in text_lower:
            return True # NGワードが1つでも入っていたらネタバレ(True)
    return False # 1つも無ければ安全(False)

# 3. 関数を全レビューに適用し、AIの予測結果として新しい列に保存する
df_sample['rule_prediction'] = df_sample['review_text'].apply(check_spoiler_rule)

# 4. 実際の正解(is_spoiler)と、予測(rule_prediction)を見比べて正解率を計算
accuracy = accuracy_score(df_sample['is_spoiler'], df_sample['rule_prediction'])
print(f"【結果】ルールベース手法の正解率: {accuracy * 100:.2f}%")

【結果】ルールベース手法の正解率: 60.90%


In [13]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# --- 1. 単語に背番号をつける（トークン化） ---
# レビューによく出てくる上位1万語だけを覚える辞書を作ります
max_words = 10000 
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df_sample['review_text'])

# 全てのレビュー文章を、単語の背番号の配列（数字のリスト）に変換します
# 例：「I love movie」 -> [12, 45, 8]
sequences = tokenizer.texts_to_sequences(df_sample['review_text'])

# --- 2. 文章の長さを揃える（パディング） ---
# レビューは長いものも短いものもあるため、AIが計算しやすいように長さを固定（今回は200語）します。
# 短い場合は0で埋め、長い場合はカットします。
max_length = 200
X = pad_sequences(sequences, maxlen=max_length)

# 正解ラベル（ネタバレTrue/Falseを、1と0の数字に変換）
y = df_sample['is_spoiler'].astype(int).values

# --- 3. 学習用とテスト用に分割 ---
# データの8割をAIの「学習用」に、残り2割を最後に実力を測る「テスト用」に分けます
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("学習用データの形:", X_train.shape)
print("テスト用データの形:", X_test.shape)

学習用データの形: (400000, 200)
テスト用データの形: (100000, 200)


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# --- 1. AIモデルの設計図を作成 ---
model = Sequential()

# 第1層：Embedding層（Word2Vecのような機能）
# 単語の背番号を、意味を持った「ベクトル（多次元の数字）」に変換します。
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_length))

# 第2層：LSTM層（ここが関所です！）
# 64個の記憶セルを持ち、過去の単語の文脈（否定や逆接）を維持・忘却しながら読み進めます。
model.add(LSTM(64))

# 第3層：出力層
# 最後に「ネタバレである確率（0〜1）」を出力します。
model.add(Dense(1, activation='sigmoid'))

# モデルのコンパイル（学習のルール設定）
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("=== LSTMモデルの設計図 ===")
model.summary()

# --- 2. いざ、学習スタート！ ---
print("\n学習を開始します...（数分かかる場合があります）")
history = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

# --- 3. ルールベースとの比較（最終テスト） ---
loss, lstm_accuracy = model.evaluate(X_test, y_test)
print(f"\n【最終結果】LSTMモデルの正解率: {lstm_accuracy * 100:.2f}%")

=== LSTMモデルの設計図 ===


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


学習を開始します...（数分かかる場合があります）
Epoch 1/5
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 384s 77ms/step - accuracy: 0.7466 - loss: 0.5234 - val_accuracy: 0.7666 - val_loss: 0.4896
Epoch 2/5
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 382s 76ms/step - accuracy: 0.7769 - loss: 0.4773 - val_accuracy: 0.7767 - val_loss: 0.4798
Epoch 3/5
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 384s 77ms/step - accuracy: 0.7893 - loss: 0.4568 - val_accuracy: 0.7726 - val_loss: 0.4823
Epoch 4/5
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 385s 77ms/step - accuracy: 0.8031 - loss: 0.4336 - val_accuracy: 0.7669 - val_loss: 0.4985
Epoch 5/5
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 384s 77ms/step - accuracy: 0.8174 - loss: 0.4080 - val_accuracy: 0.7636 - val_loss: 0.5165
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 40s 13ms/step - accuracy: 0.7620 - loss: 0.5207

【最終結果】LSTMモデルの正解率: 76.20%
